# Agente Orientador de Discentes do BTI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cleziojr/atividade_1_ia_agentica/blob/main/agente_orientador.ipynb)

### Como executar

**Colab:** adicione sua chave de api em *Secrets* (ícone de chave na barra lateral) e execute as células. A célula de setup clona o repositório e instala as dependências.

**VS Code:** crie um ambiente com Python 3.12, copie `.env.example` para `.env` com sua chave e selecione o kernel `.venv`:

```bash
uv venv --python 3.12 .venv
uv pip install -r requirements.txt
```

## IMPLEMENTAÇÃO

### CONFIGURAÇÃO E IMPORTAÇÃO DE MÓDULOS

In [39]:
import os
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    if not os.path.exists("atividade_1_ia_agentica"):
        !git clone -q https://github.com/cleziojr/atividade_1_ia_agentica.git
    %cd atividade_1_ia_agentica
    %pip install -q -r requirements.txt

In [40]:
import os
import json
import urllib.request
import shutil
import subprocess
from pathlib import Path

from pydantic import BaseModel, Field

from agentkit.model import LLMAPI
from agentkit.agent import Agent
from agentkit.tools import tool, run_python

In [41]:
if EM_COLAB:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    # Descomente o que for necessário
    # os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    # os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
else:
    from dotenv import load_dotenv

    load_dotenv()

In [42]:
# Modifique de acordo com o modelo que utilizará
llm = LLMAPI("gemini-3.8-flash", api_key=os.environ["GOOGLE_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", temperature=0.0, max_tokens=4000)

### DEFINIÇÃO DE TOOLS

In [43]:
from ferramentas_materias import comparar_materias_aluno

# Descomente e adicione em TOOLS quando implementar (o @tool exige docstring)
# @tool
# def carrega_dados_disciplinas(): # -> list:
#   pass

# @tool
# def classifica_intencao_usuario():
#   pass

# @tool
# def recomenda_grade_disciplinas():
#   pass

# @tool
# def coleta_feedback():
#   pass

# @tool
# def calcula_carga_hora_disciplinas():
#   pass

In [44]:
TOOLS = [comparar_materias_aluno]

### CLASSES E FUNÇÕES DE APOIO

#### PLANEJAMENTO

In [45]:
from agentkit.tools import render_tools

class Passo(BaseModel):
    instrucao: str
    resultado_esperado: str

class Planejamento(BaseModel):
  passos: list[Passo] = Field(min_length=2)

PLANNER_SYSTEM = f"""
Você escreve uma sequência de instruções e resultados esperados para um agente.
O agente só consegue agir por meio destas ferramentas:
{render_tools(TOOLS)}

Cada passo deve ser realizável com essas ferramentas ou com a conversa com o discente
(para obter informações que ele ainda não forneceu).
Ainda não escreva código.
"""

def cria_planejamento(tarefa: str) -> Planejamento:
  """Pede o plano ao modelo, com o papel no sistema e a tarefa no usuário."""
  response = llm.generate_structured(
        [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": tarefa}],
        Planejamento,
        max_tokens=4000,
    )
  return response

In [46]:
TAREFA = """
Elabore um planejamento de grade curricular para um discente que:

a) pode ou não saber quais disciplinas estão sendo ofertadas no semestre vigente;
b) pode ou não saber em quais disciplinas ele pode solicitar matrícula;

E que precisa saber quais:

c) disciplinas quer priorizar: componentes curriculares obrigatórios ou optativos;
d) são os dias e horários mais favoráveis para cursar disciplinas;
e) compromissos extracurriculares (estágio, projeto de pesquisa, esporte, grêmio, consultas médicas etc) bloqueiam dias e horários;
f) modalidades de disciplinas almeja priorizar (presencial, híbrido ou remoto?)
g) áreas de interesse almeja conhecer e/ou aprofundar: algoritmos e estruturas de dados? redes de computadores? ciência de dados? matemática pura? matemática aplicada?
h) docentes ofertam cada disciplina na grade proposta

E requer, ainda, que:

i) saiba qual é a carga horária almejada (quanto maior a carga horária, mais disciplinas, mais desafiante conciliar com outros compromissos e dedicar tempo para os estudos, mas não é impossível.)

---

O agente deve entregar o planejamento no tipo de arquivo e formato que o usuário solicitar, por exemplo: XML, YAML, Markdown etc.
Atente-se à linguagem do usuário, caso seja informal: pode ser .txt. Caso contrário, prepare um .pptx.
"""

planejamento = cria_planejamento(TAREFA)

for indice, passo in enumerate(planejamento.passos, start=1):
    print(f"{indice}. {passo.instrucao}")
    print(f"   espera: {passo.resultado_esperado}")

1. Interagir com o discente para solicitar seu identificador de matrícula (aluno_id) e coletar suas preferências e restrições: priorização entre matérias obrigatórias ou optativas, dias e horários livres, compromissos extracurriculares que geram conflito, modalidade pretendida (presencial, híbrida ou remota), áreas de interesse para aprofundamento, meta de carga horária e formato de arquivo desejado para o documento final, prestando atenção à formalidade da linguagem utilizada.
   espera: Identificador do discente obtido juntamente com o levantamento completo das preferências acadêmicas, restrições de agenda, carga horária pretendida e formato/tom para a entrega do documento.
2. Chamar a ferramenta comparar_materias_aluno passando o aluno_id obtido para verificar a relação de disciplinas ofertadas no semestre vigente e saber quais estão liberadas ou bloqueadas para matrícula.
   espera: Relatório das disciplinas ofertadas no semestre com status de aptidão do discente (quais pode cursar

#### EXECUÇÃO

## RELATÓRIO

Sobre o uso de ReAct ou planejamento para a tarefa que propusemos que o agente cumpra (sugestão de grades curriculares para discentes), convém ressaltar que inicialmente a abordagem utilizando o planejamento aparenta ser mais atrativa, tendo em vista que permite que o agente não realize uma sequência de perguntas ao estudante até o próprio usuário saber que ainda não possui uma determinada informação. Um exemplo real desse cenário consiste em relação ao usuário manifestar profundo interesse em relação a uma determinada disciplina, mas antes do agente orientador elaborar próximos passos, questionar ao aluno se ele cumpre os pré-requsisitos exigidos para cursá-la. Ou seja, o agente já possui um grande contexto prévio, pois conhece a sequência de etapas necessárias para cumprir a tarefa que recebeu. No ReAct, isso não necessariamente ocorre dado que "[...] ele decide o próximo passo a cada rodada, a partir do resultado do passo anterior". Em suma, continua sendo uma abordagem válida, mas o fato do agente saber toda a sequência de passos a serem executados, para o contexto atual, aparenta ser mais estratégico (falta validar ainda).